# Live Paper Trading — Daily Runner
**Run this every morning before 9:15 AM.**

Cells to run:
1. **Cell 1** — Setup: imports, sleep prevention, check token cache
2. **Cell 2** — Only if Cell 1 says "Browser opened": paste your request_token here
3. **Cell 3** — Start the agent (runs until 3:15 PM, output streams here)
4. **Cell 4** — Cleanup: turn off sleep prevention (run after market closes)


In [ ]:
1+1

In [ ]:
# ── Cell 1: Setup — run this first ──────────────────────────────────────────
import ctypes, hashlib, json, os, sys
from datetime import date
from pathlib import Path
from IPython.display import display, HTML

# Notebook lives in notebooks/ — project root is one level up
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print(f"Working directory: {PROJECT_ROOT}")

# ── Windows sleep prevention ─────────────────────────────────────────────────
ES_CONTINUOUS      = 0x80000000
ES_SYSTEM_REQUIRED = 0x00000001
ctypes.windll.kernel32.SetThreadExecutionState(ES_CONTINUOUS | ES_SYSTEM_REQUIRED)
print("Sleep prevention: ON")

# ── Load credentials ─────────────────────────────────────────────────────────
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")
API_KEY    = os.getenv("KITE_API_KEY", "")
API_SECRET = os.getenv("KITE_API_SECRET", "")
CACHE      = PROJECT_ROOT / "checkpoints" / "access_token.json"

ACCESS_TOKEN = None

# ── Check token cache ─────────────────────────────────────────────────────────
if CACHE.exists():
    _cached = json.loads(CACHE.read_text())
    if _cached.get("date") == str(date.today()):
        ACCESS_TOKEN = _cached["access_token"]
        print("✓ Cached token found for today — skip to Cell 3")

if ACCESS_TOKEN is None:
    # Build login URL directly — no kiteconnect import needed
    LOGIN_URL = f"https://kite.trade/connect/login?api_key={API_KEY}&v=3"
    display(HTML(f"""
        <div style="padding:12px; background:#f0f8ff; border-left:4px solid #1a73e8; font-size:14px;">
            <b>Step 1 — Log in to Zerodha:</b><br><br>
            <a href="{LOGIN_URL}" target="_blank" style="font-size:15px;">
                🔗 Click here to open Zerodha login
            </a><br><br>
            After login, your browser redirects to:<br>
            <code>https://127.0.0.1/?request_token=<b>XXXXXX</b>&action=login&status=success</code><br><br>
            Copy the <b>request_token</b> value and paste it in <b>Cell 2</b>.
        </div>
    """))


: 

In [ ]:
# ── Cell 2: Paste request_token here (SKIP if Cell 1 showed cached token) ───
#
# After logging in on Zerodha, your browser redirects to:
#   https://127.0.0.1/?request_token=XXXXX&action=login&status=success
# Copy the value of 'request_token' and paste it below (replace PASTE_HERE).

REQUEST_TOKEN = "PASTE_HERE"   # <── paste your request_token between the quotes

# --- run from here ---
if REQUEST_TOKEN == "PASTE_HERE":
    print("⚠  Replace PASTE_HERE with your actual request_token above, then run this cell again.")
else:
    from data_pipeline.kite_auth import login_step2
    login_step2(REQUEST_TOKEN)
    _cached = json.loads(CACHE.read_text())
    ACCESS_TOKEN = _cached["access_token"]
    print(f"✓ Login successful — access token saved")


In [ ]:
# ── Cell 3: Start the live agent ─────────────────────────────────────────────
#
# This cell BLOCKS until 3:15 PM — all live output streams here.
# Do NOT interrupt it unless you want to stop trading for the day.
# If interrupted mid-trade, the open position is saved in
#   checkpoints/live_open_trade.json — re-run this cell to resume monitoring.

if ACCESS_TOKEN is None:
    print("⚠  ACCESS_TOKEN is not set. Run Cell 1 and Cell 2 first.")
else:
    import sys
    sys.argv = ["agent.py", "--token", ACCESS_TOKEN]
    from live.agent import main as _agent_main
    _agent_main()


In [ ]:
# ── Cell 4: Cleanup — run after market closes (3:15 PM+) ────────────────────
ctypes.windll.kernel32.SetThreadExecutionState(ES_CONTINUOUS)
print("Sleep prevention: OFF")
print("\nToday's trades saved to: data/trade_logs/live_paper_trades.csv")

# Quick view of today's trade
import pandas as pd
from config.settings import TRADE_LOG_DIR
_f = TRADE_LOG_DIR / "live_paper_trades.csv"
if _f.exists():
    _df = pd.read_csv(_f)
    _today = str(date.today())
    _today_trades = _df[_df["date"] == _today]
    if not _today_trades.empty:
        display(_today_trades[["date","symbol","driver_strategy","entry_price",
                                "target","stop_loss","exit_price","exit_reason",
                                "result","pnl_rs"]].to_string(index=False))
    else:
        print("No trades placed today.")
else:
    print("No live_paper_trades.csv yet.")
